# Orientation: meet the shop and the model

This course builds an operations-desk agent for a small outdoor shop.

In this notebook, you will:

- make one model call;
- see its tokens and cost;
- see how the cache makes a repeated call fast;
- explore the shop data;
- build a small policy search; and
- meet the tickets used to test the agent.

## Setup

Run this cell first.

In [ ]:
import json
import math
import os
import re
import time
from collections import Counter
from getpass import getpass
from pathlib import Path

import litellm
import pandas as pd
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent

load_dotenv(course_root / ".env")
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
TEMPERATURE = 0
litellm.drop_params = True
litellm.cache = litellm.Cache(
    type="disk",
    disk_cache_dir=str(course_root / ".litellm_cache"),
)

## 1. Make a model call

We send one user message. The answer comes back inside a response object.

In [ ]:
response = litellm.completion(
    model=MODEL,
    temperature=TEMPERATURE,
    messages=[{
        "role": "user",
        "content": "What does the operations desk of a small online shop do? Use one sentence.",
    }],
)

print(response.choices[0].message.content)
print("tokens:", response.usage.prompt_tokens, "in,", response.usage.completion_tokens, "out")
print("cost: $", response._hidden_params.get("response_cost"), sep="")

## 2. Repeat a call from the cache

The first call below goes to the provider. The second can reuse the saved response.

A cache is useful while learning: the same request becomes faster and does not add model cost.

In [ ]:
question = "Name three things that can go wrong with a parcel."

for label, options in [
    ("fresh", {"cache": {"no-cache": True}}),
    ("repeat", {}),
]:
    started = time.perf_counter()
    result = litellm.completion(
        model=MODEL,
        temperature=TEMPERATURE,
        messages=[{"role": "user", "content": question}],
        **options,
    )
    seconds = time.perf_counter() - started
    print(label, "seconds:", round(seconds, 3), "cache hit:", result._hidden_params.get("cache_hit"))

## 3. Meet Larkspur Outfitters

The course includes a small, fixed world: products, customers, orders, policies, and return tickets.

In [ ]:
from shoplab import world

products = world.load_products()
customers = world.load_customers()
orders = world.load_orders()

print("products:", len(products))
print("customers:", len(customers))
print("orders:", len(orders))

In [ ]:
catalog = pd.DataFrame(products)
catalog[["sku", "name", "category", "price_usd", "stock"]].head()

## 4. Search the policy shelf

There are only twelve policies, but an agent still needs a way to find the useful one.

In [ ]:
policies = world.load_policies()

for policy in policies:
    print(policy["id"], "-", policy["title"])

First, turn text into useful words. Common words such as the and is do not help much.

In [ ]:
COMMON = {"a", "an", "and", "for", "i", "is", "it", "of", "the", "to"}

def words(text):
    found = re.findall(r"[a-z0-9']+", text.lower())
    return [word for word in found if word not in COMMON]

print(words("The tent is opened, but I want a refund."))

A rare matching word should count more than a common matching word.

IDF gives rare words a larger weight. The formula here is log(number of documents divided by documents containing the word).

In [ ]:
policy_words = [
    set(words(policy["title"] + " " + policy["text"]))
    for policy in policies
]
document_count = Counter(
    word
    for one_policy in policy_words
    for word in one_policy
)

def small_policy_search(query, limit=2):
    query_words = set(words(query))
    scored = []

    for policy, known_words in zip(policies, policy_words):
        shared = query_words & known_words
        score = sum(
            math.log(len(policies) / document_count[word])
            for word in shared
        )
        scored.append((score, policy))

    scored.sort(key=lambda item: item[0], reverse=True)
    return [(policy["id"], round(score, 3)) for score, policy in scored[:limit]]

print(small_policy_search("opened item refund and restocking fee"))
print(small_policy_search("parcel arrived cracked"))

The course provides the finished version as world.search_policy.

In [ ]:
hits = world.search_policy("opened item refund and restocking fee")

for hit in hits:
    print(hit["id"], hit["title"], "score:", hit["score"])

## 5. Meet the tickets and gold answers

Tickets are split into train, dev, and test groups. Each ticket has a gold answer produced by the shop's rules.

Every agent in the course must return three fields: decision, policy_id, and refund_usd.

In [ ]:
tickets = world.load_tickets()

print({name: len(rows) for name, rows in tickets.items()})
print(json.dumps(tickets["train"][0], indent=2))

## 6. The model does not know the shop data

The model has not seen our catalog. If we ask for a product price, it must guess.

In [ ]:
guess = litellm.completion(
    model=MODEL,
    temperature=TEMPERATURE,
    messages=[{
        "role": "user",
        "content": "What does the Ridgeline 2P Tent cost at Larkspur Outfitters?",
    }],
).choices[0].message.content

real_price = catalog.loc[catalog["sku"] == "LK-1001", "price_usd"].item()
print("model:", guess)
print("catalog: $", real_price, sep="")

## What to remember

1. A model call returns text, token counts, and cost.
2. A disk cache can reuse an identical response.
3. The shop world is ordinary structured data.
4. Policy search ranks documents by useful shared words.
5. Gold answers let us measure an agent later.
6. A model cannot know private shop facts until we give it access.

## Small practice steps

1. Search the catalog for the most expensive product.
2. Try small_policy_search with a battery-shipping question.
3. Open a dev ticket and name the three gold-answer fields.